# 🚀 CAVAT - Remote Dev Environment Setup

**Mục đích:** Thiết lập kết nối SSH từ VS Code (Local) tới Kaggle GPU thông qua TCP Tunnel.

**Dự án:** CAVAT - Hệ thống AI Trợ giảng Toán học (Qwen 2.5 + QLoRA + Custom RAG)

**Tunnel:** Sử dụng [bore](https://github.com/ekzhang/bore) — TCP tunnel mã nguồn mở, miễn phí, không cần đăng ký.

---

### ⚡ Hướng dẫn sử dụng:
1. Đảm bảo **GPU T4** đã bật (Settings → Accelerator → GPU T4 x2)
2. Bật **Internet** (Settings → Internet → On)
3. Chạy các cell **tuần tự** từ trên xuống dưới
4. Sau Cell 3, copy lệnh SSH hiển thị và dùng để kết nối từ VS Code

## 📋 Cell 1: Kiểm tra GPU & Môi trường

In [ ]:
# ============================================
# CELL 1: KIỂM TRA GPU & MÔI TRƯỜNG HỆ THỐNG
# ============================================

import torch
import subprocess
import sys
import os

print("="*60)
print("🔍 KIỂM TRA MÔI TRƯỜNG HỆ THỐNG CAVAT")
print("="*60)

print("\n📊 [GPU STATUS]")
!nvidia-smi

print(f"\n🔧 [PYTORCH INFO]")
print(f"   PyTorch version: {torch.__version__}")
print(f"   CUDA available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"   GPU Name:        {gpu_name}")
    print(f"   GPU Memory:      {gpu_mem:.1f} GB")
    print(f"   CUDA Version:    {torch.version.cuda}")
    print(f"\n   ✅ GPU sẵn sàng cho training Qwen 2.5 + QLoRA!")
else:
    print("\n   ❌ KHÔNG TÌM THẤY GPU!")
    print("   → Settings → Accelerator → GPU T4 x2")

print(f"\n🐍 [PYTHON INFO]")
print(f"   Python version:  {sys.version}")
print(f"   Working dir:     {os.getcwd()}")
!df -h /kaggle/working | tail -1

print("\n" + "="*60)
print("✅ Kiểm tra hoàn tất - Tiếp tục Cell 2")
print("="*60)

## 🔧 Cell 2: Cài đặt SSH Server + Bore Tunnel

In [ ]:
# ============================================
# CELL 2: CÀI ĐẶT SSH SERVER + BORE TUNNEL
# ============================================

import os

print("="*60)
print("🔧 CÀI ĐẶT SSH SERVER & BORE TUNNEL")
print("="*60)

# --- SSH Server ---
print("\n📦 [1/4] Cài đặt OpenSSH Server...")
!apt-get update -qq && apt-get install -y -qq openssh-server > /dev/null 2>&1
print("   ✅ OpenSSH Server đã cài đặt")

print("\n🔐 [2/4] Cấu hình SSH Server...")
!mkdir -p /var/run/sshd
!echo 'root:cavat123' | chpasswd
!sed -i 's/#PermitRootLogin prohibit-password/PermitRootLogin yes/' /etc/ssh/sshd_config
!sed -i 's/PermitRootLogin prohibit-password/PermitRootLogin yes/' /etc/ssh/sshd_config
!sed -i 's/#PasswordAuthentication yes/PasswordAuthentication yes/' /etc/ssh/sshd_config
!sed -i 's/PasswordAuthentication no/PasswordAuthentication yes/' /etc/ssh/sshd_config
print("   ✅ SSH đã cấu hình")

print("\n🚀 [3/4] Khởi chạy SSH Server...")
!service ssh start
print("   ✅ SSH Server đang chạy")

# --- Bore Tunnel ---
print("\n🌐 [4/4] Tải bore tunnel...")
BORE_VERSION = "v0.5.2"
if not os.path.exists('./bore'):
    !wget -q https://github.com/ekzhang/bore/releases/download/{BORE_VERSION}/bore-{BORE_VERSION}-x86_64-unknown-linux-musl.tar.gz -O bore.tar.gz
    !tar xzf bore.tar.gz && rm -f bore.tar.gz && chmod +x bore
    print("   ✅ bore đã tải xong")
else:
    print("   ✅ bore đã có sẵn")
!./bore --version

print("\n" + "="*60)
print("✅ Cài đặt hoàn tất - Tiếp tục Cell 3")
print("="*60)

## 🌐 Cell 3: Khởi chạy Bore Tunnel (chạy ngầm)

In [ ]:
# ============================================
# CELL 3: KHỞI CHẠY BORE TUNNEL (BACKGROUND)
# ============================================
#
# bore chạy ngầm → cell kết thúc bình thường
# → bạn tiếp tục chạy Cell 4, 5, 6 thoải mái
#
# ============================================

import subprocess
import time
import re
import json
import threading
import os

print("="*60)
print("🌐 KHỞI CHẠY BORE SSH TUNNEL")
print("="*60)

# Kill bore cũ nếu có
print("\n🔄 [1/2] Dọn dẹp tunnel cũ...")
!pkill -f 'bore local' 2>/dev/null || true
time.sleep(1)
print("   ✅ Đã dọn dẹp")

# Ghi log ra file để đọc port
BORE_LOG = "/tmp/bore_tunnel.log"

print("\n🚇 [2/2] Mở SSH Tunnel (background)...")
print("   ⏳ Đang kết nối tới bore.pub...")

# Chạy bore BACKGROUND bằng nohup, log ra file
os.system(f"nohup ./bore local 22 --to bore.pub > {BORE_LOG} 2>&1 &")

# Chờ và đọc log để lấy port (tối đa 20 giây)
tunnel_port = None
for i in range(20):
    time.sleep(1)
    if os.path.exists(BORE_LOG):
        with open(BORE_LOG, 'r') as f:
            log_content = f.read()
        # bore output dạng: "listening at bore.pub:XXXXX"
        match = re.search(r'bore\.pub:(\d+)', log_content)
        if match:
            tunnel_port = match.group(1)
            break
    if i % 5 == 4:
        print(f"   ⏳ Vẫn đang chờ... ({i+1}s)")

if tunnel_port:
    host = "bore.pub"
    port = tunnel_port
    
    print("\n" + "🟢"*30)
    print("\n" + "="*60)
    print("   🎉 BORE TUNNEL ĐÃ SẴN SÀNG!")
    print("="*60)
    print(f"")
    print(f"   🔗 Host:     {host}")
    print(f"   🔗 Port:     {port}")
    print(f"   🔑 Password: cavat123")
    print(f"")
    print(f"   ┌──────────────────────────────────────────────────┐")
    print(f"   │  📋 LỆNH KẾT NỐI (copy vào Terminal):          │")
    print(f"   │                                                  │")
    print(f"   │  ssh root@{host} -p {port:<27s}│")
    print(f"   │                                                  │")
    print(f"   └──────────────────────────────────────────────────┘")
    print(f"")
    print(f"   📝 VS Code SSH Config (~/.ssh/config):")
    print(f"   ┌──────────────────────────────────────────────────┐")
    print(f"   │  Host kaggle-cavat                               │")
    print(f"   │      HostName {host:<35s}│")
    print(f"   │      Port {port:<38s}│")
    print(f"   │      User root                                   │")
    print(f"   │      StrictHostKeyChecking no                    │")
    print(f"   │      UserKnownHostsFile /dev/null                │")
    print(f"   └──────────────────────────────────────────────────┘")
    print("\n" + "🟢"*30)
    
    # Lưu endpoint để watchdog dùng
    with open('/tmp/bore_endpoint.json', 'w') as f:
        json.dump({'host': host, 'port': port}, f)
    
    print("\n   ✅ Tunnel chạy ngầm — tiếp tục Cell 4, 5, 6 bình thường!")
else:
    print("\n   ❌ KHÔNG THỂ KHỞI TẠO TUNNEL!")
    print(f"")
    print(f"   📋 Bore log:")
    if os.path.exists(BORE_LOG):
        with open(BORE_LOG) as f:
            print(f.read())
    print(f"")
    print(f"   🔍 Cách khắc phục:")
    print(f"   1. Kiểm tra Internet → Settings → Internet → On")
    print(f"   2. bore.pub quá tải → Chờ 1 phút, chạy lại Cell 3")
    print(f"   3. Chạy thủ công: !./bore local 22 --to bore.pub")

print("\n" + "="*60)
print("Tiếp tục Cell 4 →")
print("="*60)

## 🐕 Cell 4: Watchdog Heartbeat (Giám sát tự động)

In [ ]:
# ============================================
# CELL 4: WATCHDOG HEARTBEAT - GIÁM SÁT TỰ ĐỘNG
# ============================================

import threading
import time
import os
import subprocess
from datetime import datetime

print("="*60)
print("🐕 KHỞI CHẠY WATCHDOG HEARTBEAT")
print("="*60)

HEARTBEAT_INTERVAL = 60
MAX_SSH_RESTARTS = 5

class WatchdogMonitor:
    def __init__(self):
        self.running = True
        self.ssh_restarts = 0
        self.check_count = 0
        self.last_bore_ok = True
        
    def check_ssh(self):
        try:
            result = os.popen("service ssh status 2>&1").read()
            return "running" in result.lower() or "active" in result.lower()
        except:
            return False
    
    def check_bore(self):
        try:
            result = subprocess.run(['pgrep', '-f', 'bore local'], capture_output=True)
            return result.returncode == 0
        except:
            return False
    
    def restart_ssh(self):
        os.system("service ssh start")
        self.ssh_restarts += 1
    
    def heartbeat(self):
        while self.running:
            self.check_count += 1
            ts = datetime.now().strftime("%H:%M:%S")
            ssh_ok = self.check_ssh()
            bore_ok = self.check_bore()
            si = "✅" if ssh_ok else "❌"
            bi = "✅" if bore_ok else "❌"
            
            if ssh_ok and bore_ok:
                if self.check_count % 5 == 0:
                    print(f"[{ts}] #{self.check_count:04d} | SSH: {si} | Bore: {bi} | OK")
            else:
                print(f"[{ts}] #{self.check_count:04d} | SSH: {si} | Bore: {bi} | ⚠️")
                if not ssh_ok and self.ssh_restarts < MAX_SSH_RESTARTS:
                    print(f"   🔄 SSH → restart ({self.ssh_restarts+1}/{MAX_SSH_RESTARTS})")
                    self.restart_ssh()
                if not bore_ok and self.last_bore_ok:
                    print(f"   🚨 BORE DOWN → Chạy lại Cell 3")
            
            self.last_bore_ok = bore_ok
            time.sleep(HEARTBEAT_INTERVAL)

watchdog = WatchdogMonitor()
threading.Thread(target=watchdog.heartbeat, daemon=True).start()

print(f"\n🐕 Watchdog đang chạy (mỗi {HEARTBEAT_INTERVAL}s)")
print(f"   Dừng: watchdog.running = False")
print("\n" + "="*60)
print("✅ Tiếp tục Cell 5 →")
print("="*60)

## 📁 Cell 5: Clone Project & Git Setup

In [ ]:
# ============================================
# CELL 5: CLONE PROJECT & GIT AUTO-SAVE
# ============================================

import os

print("="*60)
print("📁 THIẾT LẬP PROJECT & GIT")
print("="*60)

GIT_REPO = "https://github.com/minhvuogdzz/MathAI-support-Project.git"
GIT_USER = "minhvuogdzz"
GIT_EMAIL = "minhvuogdzz@users.noreply.github.com"
PROJECT_DIR = "/kaggle/working/CAVAT_PROJECT"

print("\n🔧 [1/3] Cấu hình Git...")
!git config --global user.name "{GIT_USER}"
!git config --global user.email "{GIT_EMAIL}"
!git config --global init.defaultBranch main
print(f"   ✅ Git user: {GIT_USER}")

print(f"\n📦 [2/3] Clone project...")
if os.path.exists(PROJECT_DIR):
    print(f"   ℹ️  Đã tồn tại, pull latest...")
    os.chdir(PROJECT_DIR)
    !git pull origin main 2>/dev/null || echo "   ℹ️  Không có thay đổi"
else:
    clone_result = os.system(f"git clone {GIT_REPO} {PROJECT_DIR} 2>/dev/null")
    if clone_result != 0:
        print("   ℹ️  Repo trống, init local...")
        os.makedirs(PROJECT_DIR, exist_ok=True)
        os.chdir(PROJECT_DIR)
        !git init
        !git remote add origin {GIT_REPO}
    else:
        os.chdir(PROJECT_DIR)

print(f"   ✅ Project: {PROJECT_DIR}")

print(f"\n📂 [3/3] Cấu trúc:")
!find . -maxdepth 2 -not -path './.git/*' -not -name '.git' | head -20

print("\n" + "="*60)
print("✅ Tiếp tục Cell 6 →")
print("="*60)

## 💾 Cell 6: Auto-save Functions

In [ ]:
# ============================================
# CELL 6: AUTO-SAVE FUNCTIONS
# ============================================

import os, time, threading
from datetime import datetime

PROJECT_DIR = "/kaggle/working/CAVAT_PROJECT"

def save_now(message=None):
    """💾 Lưu lên GitHub ngay. Dùng: save_now() hoặc save_now('msg')"""
    os.chdir(PROJECT_DIR)
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    msg = f"{message} [{ts}]" if message else f"Auto-save {ts}"
    
    print(f"\n💾 Lưu: {msg}")
    os.system("git add -A")
    
    if not os.popen("git diff --cached --name-only").read().strip():
        print("   ℹ️  Không có thay đổi.")
        return
    
    if os.system(f'git commit -m "{msg}"') != 0:
        return
    
    if os.system("git push -u origin main") == 0:
        print("   ✅ Push thành công!")
    else:
        print("   ❌ Push lỗi → git remote set-url origin https://<TOKEN>@github.com/minhvuogdzz/MathAI-support-Project.git")

def auto_save_periodic(mins=30):
    """⏰ Tự động lưu định kỳ. Dùng: auto_save_periodic(30)"""
    def _loop():
        n = 0
        while True:
            time.sleep(mins * 60); n += 1
            save_now(f"Auto #{n}")
    threading.Thread(target=_loop, daemon=True).start()
    print(f"⏰ Auto-save: mỗi {mins} phút")

print("="*60)
print("💾 SẴN SÀNG")
print("="*60)
print(f"   save_now()          → Lưu ngay")
print(f"   save_now('msg')     → Lưu + ghi chú")
print(f"   auto_save_periodic(30) → Tự động")
print(f"   ⚠️  Gọi save_now() trước khi đóng session!")
print("\n" + "="*60)
print("🎉 SETUP HOÀN TẤT! Kết nối VS Code ngay!")
print("="*60)

---

## 📌 Quick Reference

| Hành động | Lệnh |
|-----------|-------|
| Kết nối SSH | `ssh root@bore.pub -p <PORT>` |
| Mật khẩu | `cavat123` |
| Lưu code | `save_now()` |
| Auto-save | `auto_save_periodic(30)` |
| GPU | `nvidia-smi` |
| Dừng watchdog | `watchdog.running = False` |
| Restart tunnel | Chạy lại Cell 3 |